In [1]:
import matplotlib.pylab as plt
import numpy as np
import pandas as pd
import cv2, pathlib, os
from PIL import Image

In [2]:
label_dict = {
    1: "bending",
    2: "Lying",
    3: "empty",
    4: "sitting",
    5: "standing",
    6: "crawling"
}

label_dict2 = {
    1: "standing",
    2: "sitting",
    3: "lying",
    4: "bending",
    5: "crawling",
    6: "empty"
}

In [ ]:
def get_fall_frame_index_range(labels):
    fall_frame_range = []

    frame_indices = np.arange(1, len(labels) + 1)
    labels = labels.values.squeeze(1)

    fall_frame_indices = frame_indices[labels == 3]
    fall_start_index = -1

    for i in range(len(fall_frame_indices) - 1):
        if fall_frame_indices[i + 1] - fall_frame_indices[i] > 10:
            if fall_start_index >= 0:
                fall_frame_range.append([fall_frame_indices[fall_start_index], fall_frame_indices[i]])
                fall_start_index = -1
        else:
            if fall_start_index == -1:
                fall_start_index = i

        # print(fall_start_index)

    return fall_frame_range


def slice_video(video_number, split="train", num_of_slices=3):
    labels = pd.read_csv(f"/data/kiat/fall_detection_dataset/{split}/{video_number}/labels.csv", index_col="index")
    fall_frame_range = get_fall_frame_index_range(labels)
    if len(fall_frame_range) == 0: return
    adl_frame_range = [(fall_frame_range[i - 1][1], fall_frame_range[i][0]) for i in range(1, len(fall_frame_range))]
    adl_frame_range = [(1, fall_frame_range[0][0])] + adl_frame_range
    # print(len(fall_frame_range))
    adl_frame_range = list(filter(lambda r: r[1] - r[0] >= 90, adl_frame_range))
    # print(len(fall_frame_range))
    if len(adl_frame_range) == 0: return

    for ind in range(num_of_slices):
        # default frame
        j = np.random.randint(len(adl_frame_range))
        start_index, end_index = adl_frame_range[j]
        assert end_index >= start_index + 90
        
        start_index = np.random.randint(start_index + 20, end_index - 70)
        end_index = np.random.randint(start_index + 50, end_index - 20)
        assert end_index >= start_index + 50
        write_video(split, video_number, [start_index, end_index], ind, False)

        # fall frame
        j = np.random.randint(len(fall_frame_range))
        start_index, end_index = fall_frame_range[j]
        if end_index - start_index >= 90:
            start_index = np.random.randint(max(1, start_index - 20), start_index + 20)
            end_index = np.random.randint(end_index - 20, min(end_index + 20, len(labels) + 1))
        else:
            start_index = np.random.randint(max(1, start_index - 20), start_index)
            end_index = np.random.randint(end_index, min(end_index + 20, len(labels) + 1))

        write_video(split, video_number, [start_index, end_index], ind, True)
        # fall_video_frame_range.append([start_index, end_index])


def read_frame(split, video_number, frame_index):
    path = f"/data/kiat/fall_detection_dataset/{split}/{video_number}/rgb/rgb_{frame_index:04d}.png"
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    return image

def write_video(split, video_number, frame_range, index, fall):
    frames = [read_frame(split, video_number, i) for i in range(*frame_range)]
    H, W, _ = frames[0].shape

    label_str = "adl" if fall is False else "fall"
    output_video = f'../fall_detection_augmented_dataset/{split}/{video_number}_{label_str}_{index}.mp4'
    os.makedirs(os.path.dirname(output_video), exist_ok=True)
    # output_video = f'/data/kiat/fall_detection_dataset_augmented/{split}/{video_number}_{label_str}_{index}.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = 10  # 초당 프레임 수
    video = cv2.VideoWriter(output_video, fourcc, fps, (W, H))

    for frame in frames:
        video.write(frame)

    video.release()

In [48]:
# for path in pathlib.Path("/data/kiat/fall_detection_dataset/train").glob("*"):
#     if path.is_dir():
#         slice_video(int(path.name), "train", 3)

for path in pathlib.Path("/data/kiat/fall_detection_dataset/test").glob("*"):
    if path.is_dir():
        slice_video(int(path.name), "test", 10)

IndexError: list index out of range

In [ ]:
def parse_labels(labels):
    index = labels[labels["class"] == 3].index
    index = np.array(index)
    
    fall_parts = []
    start_index = None
    end_index = None
    for i in range(1, len(index)):
        if index[i] - index[i - 1] == 1:
            if start_index is None:
                start_index = i - 1
            end_index = i
        else:
            if start_index is not None and end_index is not None:
                fall_parts.append((index[start_index], index[end_index]))
            start_index = None
            end_index = None

    return fall_parts


def is_nearly_correct(fall_index, fall_parts):
    for start, end in fall_parts:
        if start - 10 <= fall_index <= end + 10:
            return True
    return False


def compute_accuracy(labels, fall_predictions):
    labels = labels[labels["class"] == 3]
    index = labels.index
    index = np.array(index)

    fall_parts = parse_labels(labels)
    print(fall_parts)
    print(fall_predictions)
    
    correct = []
    for fall_index in fall_predictions:
        if is_nearly_correct(fall_index, fall_parts):
            correct.append(1)
        else:
            correct.append(0)

    acc = sum(correct) / len(correct)
    print(f"Acc: {acc}")

In [44]:
with open("../prediction.txt", "r") as f:
    fall_predictions = f.readlines()

fall_predictions = list(map(int, fall_predictions))
fall_predictions = np.array(fall_predictions)
compute_accuracy(labels, fall_predictions)

[(np.int64(271), np.int64(361)), (np.int64(527), np.int64(622)), (np.int64(1057), np.int64(1147))]
[ 522  619  670 1112 1308 1322 1411 1456]
Acc: 0.375
